
# AI-Driven Export Market Decision Support System

## Preliminary Data Analysis Notebook

This notebook reviews the available TradeStat, UN Comtrade, and World Bank datasets stored in the GitHub repository.

The notebook is designed to:

1. Clone the GitHub repository in Google Colab.
2. Locate and validate the available files.
3. Inspect dataset structure and data quality.
4. Combine the TradeStat files.
5. Generate preliminary descriptive analysis.
6. Produce report-ready analytical statements.
7. Prepare the data for future analysis under Research Questions 1–4.

**Repository**

`https://github.com/bharathipavithra10-cmd/AI-Export-Market-Decision-Support`


## 1. Clone the GitHub Repository

In [1]:

from pathlib import Path
import os

REPO_URL = "https://github.com/bharathipavithra10-cmd/AI-Export-Market-Decision-Support.git"
REPO_NAME = "AI-Export-Market-Decision-Support"
PROJECT_ROOT = Path("/content") / REPO_NAME

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL}
else:
    print("Repository already exists. Skipping clone.")

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())


Cloning into 'AI-Export-Market-Decision-Support'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 189 (delta 65), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 2.75 MiB | 10.58 MiB/s, done.
Resolving deltas: 100% (65/65), done.
Project root: /content/AI-Export-Market-Decision-Support
Project root exists: True


## 2. Import Required Libraries

In [2]:

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 3. Define Repository Paths

In [3]:

RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRADESTATE_DIR = RAW_DIR / "tradestate"
UN_COMTRADE_DIR = RAW_DIR / "un_comtrade"
WORLD_BANK_DIR = RAW_DIR / "WORLD_BANK"

paths = {
    "Project root": PROJECT_ROOT,
    "Raw data": RAW_DIR,
    "TradeStat": TRADESTATE_DIR,
    "UN Comtrade": UN_COMTRADE_DIR,
    "World Bank": WORLD_BANK_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}\n")


Project root: /content/AI-Export-Market-Decision-Support
Exists: True

Raw data: /content/AI-Export-Market-Decision-Support/data/raw
Exists: True

TradeStat: /content/AI-Export-Market-Decision-Support/data/raw/tradestate
Exists: True

UN Comtrade: /content/AI-Export-Market-Decision-Support/data/raw/un_comtrade
Exists: True

World Bank: /content/AI-Export-Market-Decision-Support/data/raw/WORLD_BANK
Exists: True



## 4. Locate Available Files

In [4]:

tradestate_files = sorted(TRADESTATE_DIR.rglob("*.xlsx"))
un_comtrade_files = sorted(UN_COMTRADE_DIR.glob("*.zip"))
world_bank_files = sorted(WORLD_BANK_DIR.glob("*.xlsx"))

print(f"TradeStat files found: {len(tradestate_files)}")
print(f"UN Comtrade files found: {len(un_comtrade_files)}")
print(f"World Bank files found: {len(world_bank_files)}")


TradeStat files found: 30
UN Comtrade files found: 1
World Bank files found: 1


In [5]:

print("TradeStat files")
print("-" * 80)
for file in tradestate_files:
    print(file.relative_to(PROJECT_ROOT))

print("\nUN Comtrade files")
print("-" * 80)
for file in un_comtrade_files:
    print(file.relative_to(PROJECT_ROOT))

print("\nWorld Bank files")
print("-" * 80)
for file in world_bank_files:
    print(file.relative_to(PROJECT_ROOT))


TradeStat files
--------------------------------------------------------------------------------
data/raw/tradestate/commodity_27/TradeStat-Eidb-Export-Commodity-wise-all-countries_2020_2021.xlsx
data/raw/tradestate/commodity_27/TradeStat-Eidb-Export-Commodity-wise-all-countries_2021_2022.xlsx
data/raw/tradestate/commodity_27/TradeStat-Eidb-Export-Commodity-wise-all-countries_2022_2023.xlsx
data/raw/tradestate/commodity_27/TradeStat-Eidb-Export-Commodity-wise-all-countries_2023_2024.xlsx
data/raw/tradestate/commodity_27/TradeStat-Eidb-Export-Commodity-wise-all-countries_2024_2025.xlsx
data/raw/tradestate/commodity_27/TradeStat-Eidb-Export-Commodity-wise-all-countries_2025-2026.xlsx
data/raw/tradestate/commodity_30/TradeStat-Eidb-Export-Commodity-wise-all-countries_2020_2021.xlsx
data/raw/tradestate/commodity_30/TradeStat-Eidb-Export-Commodity-wise-all-countries_2021_2022.xlsx
data/raw/tradestate/commodity_30/TradeStat-Eidb-Export-Commodity-wise-all-countries_2022_2023.xlsx
data/raw/tra

## 5. Validate Commodity and Year Coverage

In [6]:

def extract_year_period(filename):
    match = re.search(r"(20\d{2})[_-](20\d{2})", filename)
    if match:
        return f"{match.group(1)}_{match.group(2)}"
    return None

coverage_records = []

for path in tradestate_files:
    coverage_records.append({
        "commodity_folder": path.parent.name,
        "year_period": extract_year_period(path.name),
        "filename": path.name,
    })

coverage_df = pd.DataFrame(coverage_records)

display(coverage_df.head())


,commodity_folder,year_period,filename
0,commodity_27,2020_2021,TradeStat-Eidb-Export-Commodity-wise-all-count...
1,commodity_27,2021_2022,TradeStat-Eidb-Export-Commodity-wise-all-count...
2,commodity_27,2022_2023,TradeStat-Eidb-Export-Commodity-wise-all-count...
3,commodity_27,2023_2024,TradeStat-Eidb-Export-Commodity-wise-all-count...
4,commodity_27,2024_2025,TradeStat-Eidb-Export-Commodity-wise-all-count...


In [7]:

if not coverage_df.empty:
    coverage_table = pd.crosstab(
        coverage_df["commodity_folder"],
        coverage_df["year_period"]
    )
    display(coverage_table)
else:
    print("No TradeStat files were found.")


year_period,2020_2021,2021_2022,2022_2023,2023_2024,2024_2025,2025_2026
commodity_folder,,,,,,
commodity_27,1,1,1,1,1,1
commodity_30,1,1,1,1,1,1
commodity_71,1,1,1,1,2,0
commodity_84,1,1,1,1,1,1
commodity_85,1,1,1,1,1,1


In [8]:

duplicate_periods = (
    coverage_df
    .groupby(["commodity_folder", "year_period"])
    .size()
    .reset_index(name="file_count")
)

duplicate_periods = duplicate_periods[
    duplicate_periods["file_count"] > 1
]

if duplicate_periods.empty:
    print("No duplicate commodity-year files were detected.")
else:
    print("Potential duplicate commodity-year files:")
    display(duplicate_periods)


Potential duplicate commodity-year files:


,commodity_folder,year_period,file_count
16,commodity_71,2024_2025,2


## 6. Helper Functions

In [9]:

def clean_column_names(columns):
    return (
        pd.Index(columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )


def read_tabular_file(path):
    suffix = path.suffix.lower()

    try:
        if suffix in {".xlsx", ".xls"}:
            return pd.read_excel(path)

        if suffix == ".csv":
            return pd.read_csv(path, low_memory=False)

        if suffix == ".zip":
            return pd.read_csv(path, compression="zip", low_memory=False)

        raise ValueError(f"Unsupported file type: {suffix}")

    except Exception as exc:
        print(f"Could not read {path.name}: {exc}")
        return None


def data_quality_summary(df, dataset_name):
    if df is None or df.empty:
        return {
            "dataset": dataset_name,
            "rows": 0,
            "columns": 0,
            "duplicate_rows": np.nan,
            "missing_cells": np.nan,
            "missing_percentage": np.nan,
        }

    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())

    return {
        "dataset": dataset_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": missing_cells,
        "missing_percentage": (
            missing_cells / total_cells * 100 if total_cells else 0
        ),
    }


def find_column(df, keywords):
    for keyword in keywords:
        matches = [column for column in df.columns if keyword in column]
        if matches:
            return matches[0]
    return None


## 7. Inspect One File from Each Source

In [10]:

tradestate_sample = read_tabular_file(tradestate_files[0]) if tradestate_files else None
un_comtrade_sample = read_tabular_file(un_comtrade_files[0]) if un_comtrade_files else None
world_bank_sample = read_tabular_file(world_bank_files[0]) if world_bank_files else None

sample_datasets = {
    "TradeStat": tradestate_sample,
    "UN Comtrade": un_comtrade_sample,
    "World Bank": world_bank_sample,
}

for dataset_name, df in sample_datasets.items():
    print("\n" + "=" * 90)
    print(dataset_name)
    print("=" * 90)

    if df is None:
        print("No readable file was found.")
        continue

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nFirst five rows:")
    display(df.head())


Could not read TradeData_7_21_2026_1_4_20.csv.zip: File is not a zip file

TradeStat
Shape: (178, 8)

Columns:
['TradeStat->Eidb->Export->Commodity-wise-all-countries', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']

First five rows:


,TradeStat->Eidb->Export->Commodity-wise-all-countries,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,Report Generated on: 7/20/2026 - Values in US ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S.No.,Country / Region,2019-2020,2020-2021,%Growth,2019-2020,2020-2021,%Growth
2,1,AFGHANISTAN,0.31,0.84,167.87,NaN,NaN,NaN
3,2,ALBANIA,0.02,0,-85.86,NaN,NaN,NaN
4,3,ALGERIA,3.19,2.08,-34.77,NaN,NaN,NaN



UN Comtrade
No readable file was found.

World Bank
Shape: (2125, 10)

Columns:
['Country Name', 'Country Code', 'Series Name', 'Series Code', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]', '2023 [YR2023]', '2024 [YR2024]', '2025 [YR2025]']

First five rows:


,Country Name,Country Code,Series Name,Series Code,2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,"19,955,929,052.15","14,259,995,441.08","14,497,243,872.13","17,152,234,636.87","17,778,508,875.74",..
1,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,-2.35,-20.74,-6.24,2.27,1.87,..
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,510.79,356.50,357.26,413.76,416.87,..
3,Afghanistan,AFG,"Population, total",SP.POP.TOTL,39068979,40000412,40578842,41454761,42647492,43844111
4,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,5.60,5.13,13.71,-4.64,-6.60,..


## 8. Preliminary Data Quality Summary

In [11]:

quality_records = []

for dataset_name, df in sample_datasets.items():
    quality_records.append(data_quality_summary(df, dataset_name))

quality_df = pd.DataFrame(quality_records)
display(quality_df)


,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,TradeStat,178,8,0.00,575.00,40.38
1,UN Comtrade,0,0,NaN,NaN,NaN
2,World Bank,2125,10,2.00,48.00,0.23


In [12]:

for dataset_name, df in sample_datasets.items():
    if df is None or df.empty:
        continue

    missing_table = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percentage": df.isna().mean().values * 100,
    }).sort_values("missing_percentage", ascending=False)

    print(f"\nMissing-value summary for {dataset_name}")
    display(missing_table.head(20))



Missing-value summary for TradeStat


,column,missing_count,missing_percentage
6,Unnamed: 6,177,99.44
7,Unnamed: 7,177,99.44
5,Unnamed: 5,177,99.44
4,Unnamed: 4,21,11.80
3,Unnamed: 3,14,7.87
2,Unnamed: 2,8,4.49
1,Unnamed: 1,1,0.56
0,TradeStat->Eidb->Export->Commodity-wise-all-co...,0,0.00



Missing-value summary for World Bank


,column,missing_count,missing_percentage
1,Country Code,5,0.24
2,Series Name,5,0.24
6,2022 [YR2022],5,0.24
3,Series Code,5,0.24
4,2020 [YR2020],5,0.24
5,2021 [YR2021],5,0.24
8,2024 [YR2024],5,0.24
7,2023 [YR2023],5,0.24
9,2025 [YR2025],5,0.24
0,Country Name,3,0.14


## 9. Combine All TradeStat Files

In [13]:

tradestate_frames = []

for path in tradestate_files:
    df = read_tabular_file(path)

    if df is None or df.empty:
        continue

    df = df.copy()
    df.columns = clean_column_names(df.columns)

    df["commodity_folder"] = path.parent.name
    df["year_period"] = extract_year_period(path.name)
    df["source_file"] = path.name

    tradestate_frames.append(df)

if tradestate_frames:
    tradestate_all = pd.concat(
        tradestate_frames,
        ignore_index=True,
        sort=False
    )

    print("Combined TradeStat shape:", tradestate_all.shape)
    display(tradestate_all.head())
else:
    tradestate_all = pd.DataFrame()
    print("No TradeStat files were loaded.")


Combined TradeStat shape: (6326, 11)


,tradestat_eidb_export_commodity_wise_all_countries,unnamed_1,unnamed_2,unnamed_3,unnamed_4,unnamed_5,unnamed_6,unnamed_7,commodity_folder,year_period,source_file
0,Report Generated on: 7/20/2026 - Values in US ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,commodity_27,2020_2021,TradeStat-Eidb-Export-Commodity-wise-all-count...
1,S.No.,Country / Region,2019-2020,2020-2021,%Growth,2019-2020,2020-2021,%Growth,commodity_27,2020_2021,TradeStat-Eidb-Export-Commodity-wise-all-count...
2,1,AFGHANISTAN,0.31,0.84,167.87,NaN,NaN,NaN,commodity_27,2020_2021,TradeStat-Eidb-Export-Commodity-wise-all-count...
3,2,ALBANIA,0.02,0,-85.86,NaN,NaN,NaN,commodity_27,2020_2021,TradeStat-Eidb-Export-Commodity-wise-all-count...
4,3,ALGERIA,3.19,2.08,-34.77,NaN,NaN,NaN,commodity_27,2020_2021,TradeStat-Eidb-Export-Commodity-wise-all-count...


## 10. Inspect Combined TradeStat Structure

In [14]:

if not tradestate_all.empty:
    print("Combined columns:")
    print(tradestate_all.columns.tolist())

    print("\nData types:")
    display(tradestate_all.dtypes.to_frame("data_type"))
else:
    print("Combined TradeStat dataset is empty.")


Combined columns:
['tradestat_eidb_export_commodity_wise_all_countries', 'unnamed_1', 'unnamed_2', 'unnamed_3', 'unnamed_4', 'unnamed_5', 'unnamed_6', 'unnamed_7', 'commodity_folder', 'year_period', 'source_file']

Data types:


,data_type
tradestat_eidb_export_commodity_wise_all_countries,object
unnamed_1,object
unnamed_2,object
unnamed_3,object
unnamed_4,object
unnamed_5,object
unnamed_6,object
unnamed_7,object
commodity_folder,object
year_period,object


In [15]:

if not tradestate_all.empty:
    combined_quality = pd.DataFrame([
        data_quality_summary(tradestate_all, "Combined TradeStat")
    ])
    display(combined_quality)


,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,Combined TradeStat,6326,11,0,19992,28.73


## 11. Detect Important TradeStat Columns

In [16]:

if not tradestate_all.empty:
    detected_columns = {
        "country": find_column(
            tradestate_all,
            ["country", "destination", "importing_country"]
        ),
        "product": find_column(
            tradestate_all,
            ["commodity", "product", "hs_code"]
        ),
        "export_value": find_column(
            tradestate_all,
            ["value", "export_value", "usd", "million"]
        ),
        "quantity": find_column(
            tradestate_all,
            ["quantity", "qty"]
        ),
    }

    detected_columns_df = pd.DataFrame(
        detected_columns.items(),
        columns=["role", "detected_column"]
    )

    display(detected_columns_df)
else:
    detected_columns = {}


,role,detected_column
0,country,None
1,product,tradestat_eidb_export_commodity_wise_all_count...
2,export_value,None
3,quantity,None


In [17]:

# Optional manual correction

COUNTRY_COL = detected_columns.get("country")
PRODUCT_COL = detected_columns.get("product")
EXPORT_VALUE_COL = detected_columns.get("export_value")
QUANTITY_COL = detected_columns.get("quantity")

print("COUNTRY_COL:", COUNTRY_COL)
print("PRODUCT_COL:", PRODUCT_COL)
print("EXPORT_VALUE_COL:", EXPORT_VALUE_COL)
print("QUANTITY_COL:", QUANTITY_COL)


COUNTRY_COL: None
PRODUCT_COL: tradestat_eidb_export_commodity_wise_all_countries
EXPORT_VALUE_COL: None
QUANTITY_COL: None


## 12. Convert Export Values to Numeric Format

In [18]:

if not tradestate_all.empty and EXPORT_VALUE_COL:
    tradestate_all[EXPORT_VALUE_COL] = pd.to_numeric(
        tradestate_all[EXPORT_VALUE_COL]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

    display(
        tradestate_all[EXPORT_VALUE_COL]
        .describe()
        .to_frame()
        .T
    )
else:
    print("An export-value column was not detected.")


An export-value column was not detected.


## 13. Research Question 1: Export Demand and Historical Trends


**RQ1:** What are the current export demand patterns and historical trends for selected Indian export products across international markets?


In [19]:

if (
    not tradestate_all.empty
    and COUNTRY_COL
    and EXPORT_VALUE_COL
):
    top_markets = (
        tradestate_all
        .groupby(COUNTRY_COL, dropna=False)[EXPORT_VALUE_COL]
        .sum()
        .sort_values(ascending=False)
        .head(15)
        .reset_index()
    )

    display(top_markets)
else:
    top_markets = pd.DataFrame()
    print("Country or export-value column was not detected.")


Country or export-value column was not detected.


In [20]:

if not top_markets.empty:
    ax = (
        top_markets
        .sort_values(EXPORT_VALUE_COL)
        .plot(
            kind="barh",
            x=COUNTRY_COL,
            y=EXPORT_VALUE_COL,
            figsize=(10, 6),
            legend=False,
            title="Top 15 Destination Markets by Indian Export Value"
        )
    )

    ax.set_xlabel("Total Export Value")
    ax.set_ylabel("Destination Market")
    plt.tight_layout()
    plt.show()


In [21]:

if (
    not tradestate_all.empty
    and EXPORT_VALUE_COL
):
    export_by_period = (
        tradestate_all
        .groupby("year_period", dropna=False)[EXPORT_VALUE_COL]
        .sum()
        .reset_index()
        .sort_values("year_period")
    )

    display(export_by_period)

    ax = export_by_period.plot(
        x="year_period",
        y=EXPORT_VALUE_COL,
        marker="o",
        figsize=(10, 5),
        legend=False,
        title="Indian Export Value by Reporting Period"
    )

    ax.set_xlabel("Reporting Period")
    ax.set_ylabel("Total Export Value")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    export_by_period = pd.DataFrame()


In [22]:

if (
    not tradestate_all.empty
    and EXPORT_VALUE_COL
):
    export_by_commodity = (
        tradestate_all
        .groupby("commodity_folder", dropna=False)[EXPORT_VALUE_COL]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )

    display(export_by_commodity)

    ax = export_by_commodity.plot(
        kind="bar",
        x="commodity_folder",
        y=EXPORT_VALUE_COL,
        figsize=(9, 5),
        legend=False,
        title="Total Export Value by Commodity Group"
    )

    ax.set_xlabel("Commodity Group")
    ax.set_ylabel("Total Export Value")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    export_by_commodity = pd.DataFrame()


## 14. Growth Calculations

In [23]:

if not export_by_period.empty:
    export_by_period["year_over_year_growth_percent"] = (
        export_by_period[EXPORT_VALUE_COL]
        .pct_change()
        .mul(100)
    )

    display(export_by_period)


In [24]:

if len(export_by_period) >= 2:
    first_value = export_by_period.iloc[0][EXPORT_VALUE_COL]
    last_value = export_by_period.iloc[-1][EXPORT_VALUE_COL]
    number_of_periods = len(export_by_period) - 1

    if first_value > 0:
        cagr = ((last_value / first_value) ** (1 / number_of_periods) - 1) * 100
        print(f"Estimated CAGR: {cagr:.2f}%")
    else:
        cagr = np.nan
        print("CAGR could not be calculated because the first value is zero or negative.")
else:
    cagr = np.nan
    print("At least two reporting periods are required for CAGR.")


At least two reporting periods are required for CAGR.


## 15. Market Concentration in Indian Exports

In [25]:

if (
    not tradestate_all.empty
    and COUNTRY_COL
    and EXPORT_VALUE_COL
):
    market_totals = (
        tradestate_all
        .groupby(COUNTRY_COL)[EXPORT_VALUE_COL]
        .sum()
        .sort_values(ascending=False)
    )

    total_exports = market_totals.sum()

    market_share_table = (
        market_totals
        .reset_index(name="export_value")
    )

    market_share_table["export_share_percent"] = (
        market_share_table["export_value"] / total_exports * 100
    )

    display(market_share_table.head(20))

    top_5_share = market_share_table.head(5)["export_share_percent"].sum()
    top_10_share = market_share_table.head(10)["export_share_percent"].sum()

    print(f"Top 5 market share: {top_5_share:.2f}%")
    print(f"Top 10 market share: {top_10_share:.2f}%")
else:
    market_share_table = pd.DataFrame()
    top_5_share = np.nan
    top_10_share = np.nan


## 16. Preliminary Statements Generated from the Data

In [26]:

statements = []

statements.append(
    f"The repository review identified {len(tradestate_files)} TradeStat Excel files, "
    f"{len(un_comtrade_files)} UN Comtrade file, and "
    f"{len(world_bank_files)} World Bank file."
)

if not tradestate_all.empty:
    missing_percentage = (
        tradestate_all.isna().sum().sum()
        / (tradestate_all.shape[0] * tradestate_all.shape[1])
        * 100
    )

    duplicate_rows = int(tradestate_all.duplicated().sum())

    statements.append(
        f"The combined TradeStat dataset contains {tradestate_all.shape[0]:,} observations "
        f"and {tradestate_all.shape[1]:,} variables."
    )

    statements.append(
        f"The initial data-quality review identified {duplicate_rows:,} duplicate rows and "
        f"an overall missing-cell rate of {missing_percentage:.2f}%."
    )

if not top_markets.empty:
    leading_market = top_markets.iloc[0][COUNTRY_COL]
    leading_value = top_markets.iloc[0][EXPORT_VALUE_COL]

    statements.append(
        f"{leading_market} recorded the highest aggregated Indian export value among the "
        f"available destination markets, with {leading_value:,.2f} in the source dataset's "
        f"reported monetary units."
    )

if not export_by_commodity.empty:
    leading_commodity = export_by_commodity.iloc[0]["commodity_folder"]
    leading_commodity_value = export_by_commodity.iloc[0][EXPORT_VALUE_COL]

    statements.append(
        f"{leading_commodity} represented the largest commodity group in the preliminary "
        f"dataset, with an aggregated export value of {leading_commodity_value:,.2f}."
    )

if not np.isnan(cagr):
    direction = "increased" if cagr > 0 else "decreased"

    statements.append(
        f"Across the available reporting periods, total export value {direction} at an "
        f"estimated compound annual growth rate of {cagr:.2f}%."
    )

if not np.isnan(top_5_share):
    statements.append(
        f"The five largest destination markets accounted for approximately "
        f"{top_5_share:.2f}% of the total export value in the preliminary dataset."
    )

print("REPORT-READY PRELIMINARY STATEMENTS\n")

for number, statement in enumerate(statements, start=1):
    print(f"{number}. {statement}")


REPORT-READY PRELIMINARY STATEMENTS

1. The repository review identified 30 TradeStat Excel files, 1 UN Comtrade file, and 1 World Bank file.
2. The combined TradeStat dataset contains 6,326 observations and 11 variables.
3. The initial data-quality review identified 0 duplicate rows and an overall missing-cell rate of 28.73%.



## 17. Suggested Preliminary Interpretation

The following paragraph can be adapted for the capstone report after reviewing the generated results:

> A preliminary assessment was conducted to evaluate the availability, structure, and quality of the public trade and macroeconomic datasets. The repository contained TradeStat export files for the selected HS commodity groups, together with UN Comtrade and World Bank datasets. The initial review examined commodity-year coverage, file consistency, missing values, duplicate observations, data types, destination-market concentration, export trends, and commodity-level performance. The analysis provides an initial understanding of the available evidence and identifies the preprocessing steps required before the datasets are integrated into a product-country-year analytical dataset.


## 18. Inspect UN Comtrade Structure

In [27]:

if un_comtrade_files:
    un_comtrade = read_tabular_file(un_comtrade_files[0])

    if un_comtrade is not None:
        un_comtrade.columns = clean_column_names(un_comtrade.columns)

        print("UN Comtrade shape:", un_comtrade.shape)
        print("\nUN Comtrade columns:")
        print(un_comtrade.columns.tolist())

        display(un_comtrade.head())
else:
    un_comtrade = pd.DataFrame()
    print("No UN Comtrade file was found.")


Could not read TradeData_7_21_2026_1_4_20.csv.zip: File is not a zip file


## 19. Inspect World Bank Structure

In [28]:

if world_bank_files:
    world_bank = read_tabular_file(world_bank_files[0])

    if world_bank is not None:
        world_bank.columns = clean_column_names(world_bank.columns)

        print("World Bank shape:", world_bank.shape)
        print("\nWorld Bank columns:")
        print(world_bank.columns.tolist())

        display(world_bank.head())
else:
    world_bank = pd.DataFrame()
    print("No World Bank file was found.")


World Bank shape: (2125, 10)

World Bank columns:
['country_name', 'country_code', 'series_name', 'series_code', '2020_yr2020', '2021_yr2021', '2022_yr2022', '2023_yr2023', '2024_yr2024', '2025_yr2025']


,country_name,country_code,series_name,series_code,2020_yr2020,2021_yr2021,2022_yr2022,2023_yr2023,2024_yr2024,2025_yr2025
0,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,"19,955,929,052.15","14,259,995,441.08","14,497,243,872.13","17,152,234,636.87","17,778,508,875.74",..
1,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,-2.35,-20.74,-6.24,2.27,1.87,..
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,510.79,356.50,357.26,413.76,416.87,..
3,Afghanistan,AFG,"Population, total",SP.POP.TOTL,39068979,40000412,40578842,41454761,42647492,43844111
4,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,5.60,5.13,13.71,-4.64,-6.60,..


## 20. Preparation for Research Questions 2–4


The following variables will be created after the three sources are standardized and merged:

- Destination import growth
- India market share
- Supplier competition using the Herfindahl-Hirschman Index
- Number of supplier countries
- GDP
- GDP growth
- Inflation
- Population
- Exchange rate
- Subsequent export growth
- Export Market Attractiveness Score

The final analytical unit will be a **product-country-year** record.


In [29]:

required_future_variables = [
    "year",
    "hs_code",
    "product_description",
    "importing_country",
    "country_iso_code",
    "indian_export_value",
    "destination_import_value",
    "import_quantity",
    "import_growth",
    "india_market_share",
    "supplier_competition_hhi",
    "number_of_supplier_countries",
    "gdp",
    "gdp_growth",
    "inflation",
    "population",
    "exchange_rate",
    "subsequent_export_growth",
    "export_market_attractiveness_score",
]

future_variable_checklist = pd.DataFrame({
    "required_variable": required_future_variables,
    "status": "To be created during data integration"
})

display(future_variable_checklist)


,required_variable,status
0,year,To be created during data integration
1,hs_code,To be created during data integration
2,product_description,To be created during data integration
3,importing_country,To be created during data integration
4,country_iso_code,To be created during data integration
5,indian_export_value,To be created during data integration
6,destination_import_value,To be created during data integration
7,import_quantity,To be created during data integration
8,import_growth,To be created during data integration
9,india_market_share,To be created during data integration


## 21. Next Steps


1. Confirm the exact TradeStat column mappings.
2. Standardize country names and ISO codes.
3. Harmonize HS codes and reporting periods.
4. Reshape the World Bank dataset from wide to long format if necessary.
5. Extract importer, exporter, trade value, quantity, and supplier fields from UN Comtrade.
6. Merge all three sources into a product-country-year dataset.
7. Calculate import growth, India market share, HHI, and subsequent export growth.
8. Conduct correlation analysis and multiple linear regression for RQ2.
9. Conduct supplier competition and market-share analysis for RQ3.
10. Train Linear Regression, Random Forest, and XGBoost models for RQ4.
11. Compare model performance using MAE, RMSE, and MAPE.
12. Rank export markets using the Export Market Attractiveness Score.
